# String and Text Functions

## Quick notes
### What this covers
- Measuring, slicing and searching text: `length`, `substr`, `instr`, `replace`, `trim` and friends.
- Building text: `||`, `concat`, `concat_ws`, `printf` / `format`, padding.
- Taking text apart: splitting a delimited string with a recursive CTE, extracting e-mail domains and file extensions, reversing.
- Unicode, bytes vs characters, `quote()`, `COLLATE`, `GLOB`, `group_concat`, and what to do when there is no regex.
### Key points
- SQLite text positions are **1-based**. `substr('hello', 1, 2)` is `he`.
- `length(text)` counts characters; `length(blob)` and `octet_length` count bytes.
- `instr(haystack, needle)` returns the position of the first match, or `0` for no match. It is case-sensitive.
- `trim`, `ltrim`, `rtrim` remove spaces by default, or any characters you list as a second argument.
- Most text functions given `NULL` return `NULL`, and so does `||`. Wrap with `coalesce` when that is not what you want. (`concat`, `concat_ws`, `printf` and `quote` are the exceptions, shown below.)
- There is no `split`, no `lpad`, no `reverse`, no `regexp` out of the box. Each has a small workaround shown below.
### Gotchas (the part worth revising twice)
- `trim(x, 'abc')` removes any of the **characters** a, b, c from the ends. It does not remove the word "abc".
- `LIKE` is case-insensitive for ASCII only, and `_` and `%` are wildcards. A literal `50%` needs `ESCAPE`.
- `length()` stops at the first NUL character for text, and counts bytes for a BLOB. Do not trust it on binary data cast to text.
- `substr` with a negative start counts from the end. `substr(x, 0, 3)` returns only 2 characters (position 0 does not exist).
- `upper` / `lower` change ASCII letters only. `upper('é')` stays `é` unless the ICU extension is loaded.
- `printf('%s', NULL)` gives an empty string, but `'a' || NULL` gives NULL.
### When it matters
- Cleaning imported CSV data, matching names and e-mails, building labels and reports, and parsing "tags in one column" data you inherited.
### One-line recap example
```sql
SELECT upper(trim('  ada  ')) || ' (' || length(trim('  ada  ')) || ')';   -- ADA (3)
```

## 01. Seed data
- A small, messy contacts table, like something imported from a spreadsheet.
- Some names have stray spaces, one e-mail has capital letters, one phone number has dashes, and tags are stored as comma-separated text.
- One row has a `NULL` nickname so we can see how `NULL` travels through text functions.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE contacts (
    id       INTEGER PRIMARY KEY,
    name     TEXT NOT NULL,
    nickname TEXT,
    email    TEXT NOT NULL,
    phone    TEXT,
    tags     TEXT
);
INSERT INTO contacts VALUES
 (1, '  Ada Lovelace ',  'Countess', 'Ada.Lovelace@Example.com', '555-0101', 'vip,math,uk'),
 (2, 'Alan Turing',      NULL,       'alan@bletchley.org',       '555-0102', 'math,crypto'),
 (3, 'Grace Hopper',     'Amazing',  'grace@navy.mil',           '555 0103', 'vip,compilers'),
 (4, 'Édith Piaf',       'La Môme',  'edith@paris.fr',           NULL,       NULL),
 (5, 'Linus_Torvalds',   'Tux',      'linus@kernel.org',         '555-0105', 'linux,vip');
SELECT id, name, nickname, email FROM contacts ORDER BY id;

id | name            | nickname | email                   
---+-----------------+----------+-------------------------
1  |   Ada Lovelace  | Countess | Ada.Lovelace@Example.com
2  | Alan Turing     | NULL     | alan@bletchley.org      
3  | Grace Hopper    | Amazing  | grace@navy.mil          
4  | Édith Piaf      | La Môme  | edith@paris.fr          
5  | Linus_Torvalds  | Tux      | linus@kernel.org        
(5 rows)



## 02. length, substr, instr
- `length(x)` is the number of characters. `substr(x, start, count)` is 1-based; leave off `count` to go to the end.
- A negative `start` counts from the right: `substr(x, -3)` is the last three characters.
- `instr(x, y)` gives the 1-based position of the first `y` in `x`, or `0` if absent.
- Note the raw name of id 1 has two spaces in front and one behind, so its length is 15, not 12.

In [3]:
%%sql
SELECT id, name,
       length(name)            AS len,
       substr(name, 1, 3)      AS first3,
       substr(name, -3)        AS last3,
       instr(name, ' ')        AS first_space
FROM contacts
ORDER BY id;

id | name            | len | first3 | last3 | first_space
---+-----------------+-----+--------+-------+------------
1  |   Ada Lovelace  | 15  |   A    | ce    | 1          
2  | Alan Turing     | 11  | Ala    | ing   | 5          
3  | Grace Hopper    | 12  | Gra    | per   | 6          
4  | Édith Piaf      | 10  | Édi    | iaf   | 6          
5  | Linus_Torvalds  | 14  | Lin    | lds   | 0          
(5 rows)



## 03. substr edge cases
- Position `0` does not exist: `substr('hello', 0, 3)` gives only `he` (two characters) because the window starts one step before the string.
- A count that runs past the end is fine; you just get what is left.
- A negative count takes characters **before** the start (SQLite 3.8+). It is rare; just know it exists.
- `NULL` in, `NULL` out.

In [4]:
%%sql
SELECT substr('hello', 0, 3)   AS start_zero,
       substr('hello', 1, 3)   AS start_one,
       substr('hello', 4, 99)  AS count_too_big,
       substr('hello', 4, -2)  AS negative_count,
       substr('hello', 10)     AS past_end,
       substr(NULL, 1, 2)      AS null_in;

start_zero | start_one | count_too_big | negative_count | past_end | null_in
-----------+-----------+---------------+----------------+----------+--------
he         | hel       | lo            | el             |          | NULL   
(1 row)



## 04. upper, lower, and the ASCII-only catch
- `upper` and `lower` convert **ASCII** letters only. Accented letters are left alone unless SQLite was built with ICU.
- Compare `Édith Piaf`: `upper` gives `ÉDITH PIAF` only because the `É` was already a capital. `lower` gives `Édith piaf`: the `É` is not lowered.
- Practical rule: if you must compare names across accents or cases, normalise in your application, or store a separate normalised column.

In [5]:
%%sql
SELECT name,
       upper(name) AS up,
       lower(name) AS low
FROM contacts
WHERE id IN (3, 4)
ORDER BY id;

name         | up           | low         
-------------+--------------+-------------
Grace Hopper | GRACE HOPPER | grace hopper
Édith Piaf   | ÉDITH PIAF   | Édith piaf  
(2 rows)



## 05. trim, ltrim, rtrim
- With one argument they remove **spaces** from the ends. `ltrim` = left only, `rtrim` = right only.
- With a second argument they remove **any character in that set**, not a word.
- Trap: `trim('xxhixx', 'x')` is fine, but `trim('abcHELLOcba', 'abc')` strips every a, b, c from both ends, so you get `HELLO`; and `trim('banana', 'ban')` strips everything.
- Tabs and newlines are not spaces: trim them by listing `char(9, 10, 13)` explicitly.

In [6]:
%%sql
SELECT '[' || name || ']'               AS raw,
       '[' || trim(name) || ']'         AS trimmed,
       '[' || ltrim(name) || ']'        AS left_only,
       '[' || rtrim(name) || ']'        AS right_only
FROM contacts
WHERE id = 1;
SELECT trim('abcHELLOcba', 'abc') AS char_set_trim,
       trim('banana', 'ban')      AS eats_everything,
       '[' || trim(char(9) || ' x ' || char(10), ' ' || char(9, 10, 13)) || ']' AS tabs_and_newlines;

raw               | trimmed        | left_only       | right_only      
------------------+----------------+-----------------+-----------------
[  Ada Lovelace ] | [Ada Lovelace] | [Ada Lovelace ] | [  Ada Lovelace]
(1 row)

char_set_trim | eats_everything | tabs_and_newlines
--------------+-----------------+------------------
HELLO         |                 | [x]              
(1 row)



## 06. replace
- `replace(x, find, with)` replaces **every** occurrence, case-sensitively.
- Replacing with `''` deletes. This is the standard way to strip characters.
- Chain calls to clean several things: `replace(replace(phone, '-', ''), ' ', '')`.
- If `find` is the empty string, the original is returned unchanged.
- Counting trick: `length(x) - length(replace(x, ',', ''))` is the number of commas.

In [7]:
%%sql
SELECT id, phone,
       replace(replace(phone, '-', ''), ' ', '') AS digits_only,
       replace('aaa', '', 'X')                   AS empty_find
FROM contacts
ORDER BY id;
SELECT id, tags,
       length(tags) - length(replace(tags, ',', '')) + 1 AS tag_count
FROM contacts
WHERE tags IS NOT NULL
ORDER BY id;

id | phone    | digits_only | empty_find
---+----------+-------------+-----------
1  | 555-0101 | 5550101     | aaa       
2  | 555-0102 | 5550102     | aaa       
3  | 555 0103 | 5550103     | aaa       
4  | NULL     | NULL        | aaa       
5  | 555-0105 | 5550105     | aaa       
(5 rows)

id | tags          | tag_count
---+---------------+----------
1  | vip,math,uk   | 3        
2  | math,crypto   | 2        
3  | vip,compilers | 2        
5  | linux,vip     | 2        
(4 rows)



## 07. Concatenation: ||, concat, concat_ws
- `||` joins text. Any `NULL` makes the whole result `NULL`.
- `concat(a, b, ...)` (SQLite 3.44+) treats `NULL` as empty text.
- `concat_ws(sep, a, b, ...)` joins with a separator and **skips** `NULL`s, so you never get a dangling separator.
- Numbers are converted to text automatically.
- Basic `||` is also shown in 01-Basics/02-Filtering-And-Operators.

In [8]:
%%sql
SELECT id,
       trim(name) || ' aka ' || nickname            AS with_pipes,
       concat(trim(name), ' aka ', nickname)        AS with_concat,
       concat_ws(' aka ', trim(name), nickname)     AS with_ws,
       trim(name) || ' aka ' || coalesce(nickname, '(none)') AS with_coalesce
FROM contacts
ORDER BY id;

id | with_pipes                | with_concat               | with_ws                   | with_coalesce            
---+---------------------------+---------------------------+---------------------------+--------------------------
1  | Ada Lovelace aka Countess | Ada Lovelace aka Countess | Ada Lovelace aka Countess | Ada Lovelace aka Countess
2  | NULL                      | Alan Turing aka           | Alan Turing               | Alan Turing aka (none)   
3  | Grace Hopper aka Amazing  | Grace Hopper aka Amazing  | Grace Hopper aka Amazing  | Grace Hopper aka Amazing 
4  | Édith Piaf aka La Môme    | Édith Piaf aka La Môme    | Édith Piaf aka La Môme    | Édith Piaf aka La Môme   
5  | Linus_Torvalds aka Tux    | Linus_Torvalds aka Tux    | Linus_Torvalds aka Tux    | Linus_Torvalds aka Tux   
(5 rows)



## 08. printf / format
- `printf(fmt, args...)` is C-style formatting. `format` is a newer alias for the same function.
- `%s` text, `%d` integer, `%f` real, `%05d` zero padded, `%.2f` two decimals, `%-10s` left-aligned in 10 characters, `%10s` right-aligned, `%%` a literal percent sign.
- `%s` with `NULL` prints an empty string, which is different from `||`.
- `%d` truncates a real towards zero; it does not round.
- Width counts **bytes** here, so `Édith Piaf` (11 bytes) gets one space fewer than its 10 characters suggest. Padded columns with accents look misaligned (see section 09).

In [9]:
%%sql
SELECT printf('%05d', 42)            AS zero_padded,
       printf('%.2f', 3.14159)       AS two_decimals,
       printf('%-8s|', 'ab')         AS left_aligned,
       printf('%8s|', 'ab')          AS right_aligned,
       printf('%d%%', 85)            AS percent_sign,
       printf('%d', 7.9)             AS d_truncates,
       printf('[%s]', NULL)          AS null_is_empty,
       format('%s-%s', 'a', 'b')     AS format_alias;

zero_padded | two_decimals | left_aligned | right_aligned | percent_sign | d_truncates | null_is_empty | format_alias
------------+--------------+--------------+---------------+--------------+-------------+---------------+-------------
00042       | 3.14         | ab      |    |       ab|     | 85%          | 7           | []            | a-b         
(1 row)



## 09. Padding (there is no lpad / rpad)
- `printf('%5s', x)` and `printf('%-5s', x)` pad with spaces.
- To pad with another character, use `substr` on a repeated string: `substr('00000' || x, -5)` keeps the last 5 characters.
- `replace(hex(zeroblob(n)), '00', '*')` builds a string of `n` copies of `*`. A handy "repeat" function.
- Row 4 (`Édith Piaf`) comes out one space short because `printf` pads by bytes, not characters.
- Do not pad in SQL for storage. Pad for display and let your application own the final look.

In [10]:
%%sql
SELECT id,
       printf('%-14s|', trim(name))                AS name_padded,
       substr('00000' || id, -5)                   AS id_zero5,
       replace(hex(zeroblob(id)), '00', '*')       AS stars_by_id,
       substr(trim(name) || '..............', 1, 14) AS dot_leader
FROM contacts
ORDER BY id;

id | name_padded     | id_zero5 | stars_by_id | dot_leader    
---+-----------------+----------+-------------+---------------
1  | Ada Lovelace  | | 00001    | *           | Ada Lovelace..
2  | Alan Turing   | | 00002    | **          | Alan Turing...
3  | Grace Hopper  | | 00003    | ***         | Grace Hopper..
4  | Édith Piaf   |  | 00004    | ****        | Édith Piaf....
5  | Linus_Torvalds| | 00005    | *****       | Linus_Torvalds
(5 rows)



## 10. LIKE vs instr
- `LIKE '%x%'` and `instr(col, 'x') > 0` both answer "does it contain x?".
- `LIKE` is case-insensitive for ASCII and treats `%` and `_` as wildcards. `instr` is case-sensitive and takes the needle literally.
- So searching for a literal `_` (as in `Linus_Torvalds`) with `LIKE '%_%'` matches **every** row. `instr(name, '_') > 0` matches only the right one.
- To use `LIKE` safely, escape: `LIKE '%\_%' ESCAPE '\'`.
- Neither can use a normal index when the pattern starts with `%` (see 07-Performance).

In [11]:
%%sql
SELECT id, name,
       name LIKE '%_%'                AS like_naive,
       instr(name, '_') > 0           AS instr_literal,
       name LIKE '%\_%' ESCAPE '\'    AS like_escaped
FROM contacts
ORDER BY id;
SELECT id, name FROM contacts WHERE name LIKE '%ADA%' ORDER BY id;
SELECT id, name FROM contacts WHERE instr(name, 'ADA') > 0 ORDER BY id;

id | name            | like_naive | instr_literal | like_escaped
---+-----------------+------------+---------------+-------------
1  |   Ada Lovelace  | 1          | 0             | 0           
2  | Alan Turing     | 1          | 0             | 0           
3  | Grace Hopper    | 1          | 0             | 0           
4  | Édith Piaf      | 1          | 0             | 0           
5  | Linus_Torvalds  | 1          | 1             | 1           
(5 rows)

id | name           
---+----------------
1  |   Ada Lovelace 
(1 row)

id | name
---+-----
(0 rows)



## 11. GLOB for case-sensitive patterns
- `GLOB` uses Unix wildcards: `*` any run, `?` one character, `[abc]` a set, `[a-z]` a range, `[^0-9]` a negated set.
- It is **case-sensitive** and has no escape character; to match a literal `*` put it in brackets: `[*]`.
- It is the nearest thing to a regex in core SQLite. Use it for checks like "all digits" or "looks like a phone number".
- More on `LIKE` / `GLOB` basics in 01-Basics/02-Filtering-And-Operators.

In [12]:
%%sql
SELECT id, phone,
       phone GLOB '[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]' AS dashed_ok,
       phone GLOB '*[^0-9-]*'                              AS has_other_chars
FROM contacts
ORDER BY id;
SELECT 'Hello' GLOB 'h*' AS glob_lower_h, 'Hello' LIKE 'h%' AS like_lower_h;

id | phone    | dashed_ok | has_other_chars
---+----------+-----------+----------------
1  | 555-0101 | 1         | 0              
2  | 555-0102 | 1         | 0              
3  | 555 0103 | 0         | 1              
4  | NULL     | NULL      | NULL           
5  | 555-0105 | 1         | 0              
(5 rows)

glob_lower_h | like_lower_h
-------------+-------------
0            | 1           
(1 row)



## 12. Extracting parts: e-mail domain, user, file extension
- Combine `instr` and `substr`: user is everything before `@`, domain everything after.
- Normalise first with `lower(trim(...))`, or you will treat `Example.com` and `example.com` as different domains.
- The **last** dot of a path is harder because `instr` finds the first. For text with a single `@` or a single dot, `substr` after `instr` is enough. For the last dot, see the next section.

In [13]:
%%sql
SELECT id,
       lower(trim(email))                                                AS clean_email,
       substr(lower(trim(email)), 1, instr(trim(email), '@') - 1)              AS user_part,
       substr(lower(trim(email)), instr(trim(email), '@') + 1)                 AS domain
FROM contacts
ORDER BY id;
SELECT domain, count(*) AS n
FROM (SELECT substr(lower(trim(email)), instr(trim(email), '@') + 1) AS domain FROM contacts)
GROUP BY domain
ORDER BY n DESC, domain;

id | clean_email              | user_part    | domain       
---+--------------------------+--------------+--------------
1  | ada.lovelace@example.com | ada.lovelace | example.com  
2  | alan@bletchley.org       | alan         | bletchley.org
3  | grace@navy.mil           | grace        | navy.mil     
4  | edith@paris.fr           | edith        | paris.fr     
5  | linus@kernel.org         | linus        | kernel.org   
(5 rows)

domain        | n
--------------+--
bletchley.org | 1
example.com   | 1
kernel.org    | 1
navy.mil      | 1
paris.fr      | 1
(5 rows)



## 13. The last occurrence: file extension of a path
- We want the text after the **last** `/` or `.`, which `instr` cannot find directly.
- SQLite has no "search from the right" function, so we walk the string ourselves.
- Below: a small recursive CTE visits every position and remembers the **position of the last dot** and of the last `/`. Then `substr` takes the file name and the extension.
- A dot only counts as an extension separator if it comes after the last `/`, so `/etc/hosts` has no extension (`NULL`).
- Simplest rule of thumb: if you need this a lot, store the extension in its own column.

In [14]:
%%sql
CREATE TABLE files (id INTEGER PRIMARY KEY, path TEXT NOT NULL);
INSERT INTO files VALUES
 (1, '/home/ada/notes.txt'),
 (2, '/home/ada/archive.tar.gz'),
 (3, '/etc/hosts'),
 (4, 'report.final.pdf');
WITH RECURSIVE scan(id, path, i, last_dot, last_slash) AS (
    SELECT id, path, 1, 0, 0 FROM files
    UNION ALL
    SELECT id, path, i + 1,
           CASE WHEN substr(path, i, 1) = '.' THEN i ELSE last_dot   END,
           CASE WHEN substr(path, i, 1) = '/' THEN i ELSE last_slash END
    FROM scan
    WHERE i <= length(path)
)
SELECT id, path,
       substr(path, last_slash + 1)                                  AS file_name,
       CASE WHEN last_dot > last_slash THEN substr(path, last_dot + 1) END AS extension
FROM scan
WHERE i = length(path) + 1
ORDER BY id;

id | path                     | file_name        | extension
---+--------------------------+------------------+----------
1  | /home/ada/notes.txt      | notes.txt        | txt      
2  | /home/ada/archive.tar.gz | archive.tar.gz   | gz       
3  | /etc/hosts               | hosts            | NULL     
4  | report.final.pdf         | report.final.pdf | pdf      
(4 rows)



## 14. Splitting a delimited string with a recursive CTE
- SQLite has no `split` function. A recursive CTE peels one item off the front each step.
- Idea: keep `item` (text before the first comma) and `rest` (text after it). Append a trailing comma so the last item is not special.
- Recursion stops when `rest` is empty. The result is one row per tag, ready for `GROUP BY`, `JOIN` and counting.
- Recursive CTE basics are in 02-Querying (recursive CTEs); here we only use them as a string tool.

In [15]:
%%sql
WITH RECURSIVE split(id, item, rest) AS (
    SELECT id, '', tags || ',' FROM contacts WHERE tags IS NOT NULL
    UNION ALL
    SELECT id,
           substr(rest, 1, instr(rest, ',') - 1),
           substr(rest, instr(rest, ',') + 1)
    FROM split
    WHERE rest <> ''
)
SELECT id, item
FROM split
WHERE item <> ''
ORDER BY id, item;

id | item     
---+----------
1  | math     
1  | uk       
1  | vip      
2  | crypto   
2  | math     
3  | compilers
3  | vip      
5  | linux    
5  | vip      
(9 rows)



## 15. Using the split: tag counts and membership tests
- Once split, tags are normal rows: count how many contacts carry each tag.
- For a quick "does this row have the tag `vip`?" you do not need a split. Wrap both sides in commas: `',' || tags || ',' LIKE '%,vip,%'`. That avoids matching `lin` inside `linux`: the exact test below finds no `lin` tag, while the sloppy `LIKE '%lin%'` wrongly matches.
- `instr` version of the same: `instr(',' || tags || ',', ',vip,') > 0`.
- Long-term fix: a proper `contact_tags` table (see 09-Design). CSV-in-a-column is a trap: no foreign keys, no indexes, no constraints.

In [16]:
%%sql
WITH RECURSIVE split(id, item, rest) AS (
    SELECT id, '', tags || ',' FROM contacts WHERE tags IS NOT NULL
    UNION ALL
    SELECT id, substr(rest, 1, instr(rest, ',') - 1), substr(rest, instr(rest, ',') + 1)
    FROM split WHERE rest <> ''
)
SELECT item AS tag, count(*) AS contacts
FROM split
WHERE item <> ''
GROUP BY item
ORDER BY contacts DESC, tag;
SELECT id, name, tags,
       instr(',' || tags || ',', ',vip,') > 0 AS is_vip,
       instr(',' || tags || ',', ',lin,') > 0 AS exact_lin,
       tags LIKE '%lin%'                      AS sloppy_lin
FROM contacts
ORDER BY id;

tag       | contacts
----------+---------
vip       | 3       
math      | 2       
compilers | 1       
crypto    | 1       
linux     | 1       
uk        | 1       
(6 rows)

id | name            | tags          | is_vip | exact_lin | sloppy_lin
---+-----------------+---------------+--------+-----------+-----------
1  |   Ada Lovelace  | vip,math,uk   | 1      | 0         | 0         
2  | Alan Turing     | math,crypto   | 0      | 0         | 0         
3  | Grace Hopper    | vip,compilers | 1      | 0         | 0         
4  | Édith Piaf      | NULL          | NULL   | NULL      | NULL      
5  | Linus_Torvalds  | linux,vip     | 1      | 0         | 1         
(5 rows)



## 16. Reversing a string with recursion
- There is no `reverse()`. Build the answer one character at a time: move the first character of the input to the front of the output.
- Each step shrinks `src` by one character and grows `out`. It stops when `src` is empty.
- Same idea gives a palindrome test: a word is a palindrome when it equals its reverse.
- Only worth it for puzzles or odd reports; in real work reverse text in the application.

In [17]:
%%sql
WITH RECURSIVE rev(word, src, out) AS (
    SELECT w, w, '' FROM (SELECT 'level' AS w UNION ALL SELECT 'sqlite' UNION ALL SELECT 'noon' UNION ALL SELECT 'abc')
    UNION ALL
    SELECT word, substr(src, 2), substr(src, 1, 1) || out
    FROM rev
    WHERE src <> ''
)
SELECT word, out AS reversed, word = out AS is_palindrome
FROM rev
WHERE src = ''
ORDER BY word;

word   | reversed | is_palindrome
-------+----------+--------------
abc    | cba      | 0            
level  | level    | 1            
noon   | noon     | 1            
sqlite | etilqs   | 0            
(4 rows)



## 17. Unicode: char(), unicode(), hex()
- `unicode(x)` is the code point of the first character. `char(n, ...)` builds text from code points.
- `hex(x)` shows the bytes. For text this is the **UTF-8** encoding, so `é` (code point 233) shows as `C3A9`: two bytes.
- `char(65, 66, 67)` is `ABC`. Handy for building control characters: `char(10)` newline, `char(9)` tab.
- `unicode('')` on an empty string is `NULL`.

In [18]:
%%sql
SELECT unicode('A')                 AS code_A,
       unicode('é')                 AS code_e_acute,
       char(65, 66, 67)             AS abc,
       hex('A')                     AS hex_A,
       hex('é')                     AS hex_e_acute,
       unicode('')                  AS empty_is_null,
       'line1' || char(10) || 'line2' AS two_lines;

code_A | code_e_acute | abc | hex_A | hex_e_acute | empty_is_null | two_lines  
-------+--------------+-----+-------+-------------+---------------+------------
65     | 233          | ABC | 41    | C3A9        | NULL          | line1
line2
(1 row)



## 18. Characters vs bytes: length, octet_length, BLOBs
- `length(text)` = characters. `octet_length(text)` (3.43+) = bytes of the UTF-8 encoding. `length(blob)` = bytes.
- "Édith Piaf" is 10 characters but 11 bytes because `É` takes two bytes.
- Casting text to BLOB exposes the bytes: `length(CAST(x AS BLOB))` is the byte count.
- Text containing an embedded NUL (`char(0)`) is cut short by `length`, so never use `length` to size binary data stored as text. Store binary data as BLOB.

In [19]:
%%sql
SELECT name,
       length(name)                    AS chars,
       octet_length(name)              AS bytes,
       length(CAST(name AS BLOB))      AS blob_bytes,
       typeof(CAST(name AS BLOB))      AS cast_type
FROM contacts
WHERE id IN (1, 4)
ORDER BY id;
SELECT length('a' || char(0) || 'b')                   AS text_stops_at_nul,
       length(CAST('a' || char(0) || 'b' AS BLOB))     AS blob_counts_all,
       hex('a' || char(0) || 'b')                      AS hex_shows_all;

name            | chars | bytes | blob_bytes | cast_type
----------------+-------+-------+------------+----------
  Ada Lovelace  | 15    | 15    | 15         | blob     
Édith Piaf      | 10    | 11    | 11         | blob     
(2 rows)

text_stops_at_nul | blob_counts_all | hex_shows_all
------------------+-----------------+--------------
1                 | 3               | 610062       
(1 row)



## 19. quote() and safe literals
- `quote(x)` returns the value as a SQL literal: text gets single quotes with inner quotes doubled, `NULL` becomes the word `NULL`, numbers stay bare, blobs become `X'..'`.
- Useful for debugging ("what is *really* in this cell?") and for generating INSERT scripts.
- It shows invisible differences: `'5'` (text) vs `5` (integer) look identical in a result grid but not under `quote`.
- Never build SQL by gluing user text together; use bound parameters from your language instead.

In [20]:
%%sql
SELECT quote('O''Brien')     AS q_text,
       quote(NULL)          AS q_null,
       quote(5)             AS q_int,
       quote('5')           AS q_text_five,
       quote(2.5)           AS q_real,
       quote(x'4142')       AS q_blob;
SELECT id, quote(nickname) AS nick FROM contacts ORDER BY id;

q_text     | q_null | q_int | q_text_five | q_real | q_blob 
-----------+--------+-------+-------------+--------+--------
'O''Brien' | NULL   | 5     | '5'         | 2.5    | X'4142'
(1 row)

id | nick      
---+-----------
1  | 'Countess'
2  | NULL      
3  | 'Amazing' 
4  | 'La Môme' 
5  | 'Tux'     
(5 rows)



## 20. COLLATE: how text is compared
- Default `BINARY`: compares bytes, so `'a'` and `'A'` are different, and `'Z'` sorts before `'a'` (first list below: `Cherry`, `Zebra`, `apple`, `banana`; second, with NOCASE: `apple`, `banana`, `Cherry`, `Zebra`).
- `NOCASE`: ASCII letters compare ignoring case. `RTRIM`: ignores trailing spaces.
- You can set a collation on a column (`name TEXT COLLATE NOCASE`) or on one expression (`ORDER BY name COLLATE NOCASE`). It affects `=`, `<`, `ORDER BY`, `GROUP BY`, `DISTINCT`.
- Same ASCII-only limit as `upper` / `lower`. For real language-aware sorting you need ICU or sorting in your application.
- See also the sorting demo in 01-Basics/01-Fundamentals.

In [21]:
%%sql
SELECT 'a' = 'A'                          AS binary_eq,
       'a' = 'A' COLLATE NOCASE           AS nocase_eq,
       'abc  ' = 'abc'                    AS trailing_spaces_binary,
       'abc  ' = 'abc' COLLATE RTRIM      AS trailing_spaces_rtrim,
       'é' = 'É' COLLATE NOCASE           AS accents_not_folded;
SELECT w FROM (SELECT 'banana' AS w UNION ALL SELECT 'Zebra' UNION ALL SELECT 'Cherry' UNION ALL SELECT 'apple')
ORDER BY w;
SELECT w FROM (SELECT 'banana' AS w UNION ALL SELECT 'Zebra' UNION ALL SELECT 'Cherry' UNION ALL SELECT 'apple')
ORDER BY w COLLATE NOCASE;

binary_eq | nocase_eq | trailing_spaces_binary | trailing_spaces_rtrim | accents_not_folded
----------+-----------+------------------------+-----------------------+-------------------
0         | 1         | 0                      | 1                     | 0                 
(1 row)

w     
------
Cherry
Zebra 
apple 
banana
(4 rows)

w     
------
apple 
banana
Cherry
Zebra 
(4 rows)



## 21. Soundex and other "sounds like" caveats
- SQLite has a `soundex()` function, but only when compiled with `SQLITE_SOUNDEX`. Many builds, including most Python ones, leave it out.
- Below we try it with `expect_error` so the notebook runs either way. If it is missing you will see "no such function".
- Fuzzy matching in practice: use a normalised column (lowercase, no accents), the FTS5 extension for search (see 07-Performance), or do phonetic matching in your application.

In [22]:
%%sql expect_error
SELECT soundex('Robert') AS s1, soundex('Rupert') AS s2;

ERROR: no such function: soundex



## 22. No regex: what to use instead
- Core SQLite has no `REGEXP` function. `x REGEXP y` is parsed but fails with "no such function: REGEXP" until your application registers one.
- Python tip: `con.create_function('REGEXP', 2, lambda p, s: re.search(p, s) is not None)`. After that `WHERE col REGEXP '^a.*z$'` works.
- In pure SQL, combine: `LIKE` / `GLOB` for patterns, `instr` / `substr` for parsing, `replace` for stripping, and `CASE` for branching.
- Below: the error, then a GLOB check for "starts with a capital letter and has only letters and spaces".

In [23]:
%%sql expect_error
SELECT 'abc' REGEXP '^a';

ERROR: no such function: REGEXP



In [24]:
%%sql
SELECT id, trim(name) AS name,
       trim(name) GLOB '[A-Z]*'                                    AS starts_capital,
       trim(name) GLOB '*[^A-Za-z ]*'                              AS has_non_letters
FROM contacts
ORDER BY id;

id | name           | starts_capital | has_non_letters
---+----------------+----------------+----------------
1  | Ada Lovelace   | 1              | 0              
2  | Alan Turing    | 1              | 0              
3  | Grace Hopper   | 1              | 0              
4  | Édith Piaf     | 0              | 1              
5  | Linus_Torvalds | 1              | 1              
(5 rows)



## 23. String aggregation with group_concat
- `group_concat(x, sep)` joins the values of a group into one string. The default separator is a comma.
- `NULL` values are skipped. Order inside the string is **not guaranteed** by the old syntax; since 3.44 you can write `group_concat(x, ',' ORDER BY x)`.
- The inverse of splitting (sections 14-15): take tag rows back into one text value.
- `DISTINCT` works too, but then you cannot choose a custom separator.

In [25]:
%%sql
WITH RECURSIVE split(id, item, rest) AS (
    SELECT id, '', tags || ',' FROM contacts WHERE tags IS NOT NULL
    UNION ALL
    SELECT id, substr(rest, 1, instr(rest, ',') - 1), substr(rest, instr(rest, ',') + 1)
    FROM split WHERE rest <> ''
),
tags_rows AS (SELECT id, item FROM split WHERE item <> '')
SELECT item AS tag,
       group_concat(trim(c.name), ' / ' ORDER BY c.id) AS who
FROM tags_rows t JOIN contacts c USING (id)
GROUP BY item
ORDER BY item;

tag       | who                                         
----------+---------------------------------------------
compilers | Grace Hopper                                
crypto    | Alan Turing                                 
linux     | Linus_Torvalds                              
math      | Ada Lovelace / Alan Turing                  
uk        | Ada Lovelace                                
vip       | Ada Lovelace / Grace Hopper / Linus_Torvalds
(6 rows)



## 24. A cleaning pipeline: bad vs good
- Bad: clean in the report query every time, copy-pasting `trim`, `lower`, `replace`.
- Good: clean once while loading, or expose a view with the clean columns, so everyone sees the same result.
- Order of operations matters: trim first, then case, then replacements.
- Below: a view with a clean name, clean e-mail and digits-only phone. The second query hunts for bad phone numbers (wrong length or non-digits) and returns 0 rows, so the clean-up worked.

In [26]:
%%sql
CREATE VIEW contacts_clean AS
SELECT id,
       replace(trim(name), '_', ' ')                        AS name,
       lower(trim(email))                                   AS email,
       replace(replace(phone, '-', ''), ' ', '')            AS phone_digits
FROM contacts;
SELECT * FROM contacts_clean ORDER BY id;
SELECT id, phone_digits
FROM contacts_clean
WHERE phone_digits IS NOT NULL AND (length(phone_digits) <> 7 OR phone_digits GLOB '*[^0-9]*')
ORDER BY id;

id | name           | email                    | phone_digits
---+----------------+--------------------------+-------------
1  | Ada Lovelace   | ada.lovelace@example.com | 5550101     
2  | Alan Turing    | alan@bletchley.org       | 5550102     
3  | Grace Hopper   | grace@navy.mil           | 5550103     
4  | Édith Piaf     | edith@paris.fr           | NULL        
5  | Linus Torvalds | linus@kernel.org         | 5550105     
(5 rows)

id | phone_digits
---+-------------
(0 rows)

